# 식품안전나라 Open API 수집

이 노트북 하나로 다음 세 서비스를 수집합니다. 별도의 공통 모듈은 필요하지 않습니다.

- `I0030`: 건강기능식품 품목제조 신고사항
- `I1250`: 식품(첨가물) 품목제조보고
- `C002`: 식품(첨가물) 품목제조보고 원재료

인증키는 코드에 저장하지 않고 환경변수 `FOODSAFETYKOREA_API_KEY`에서 읽습니다. 처음에는 `RUN_COLLECTION=False` 상태로 전체 셀을 실행해 설정을 점검하세요.


In [1]:
from __future__ import annotations

import csv
from datetime import date, datetime, timedelta
import json
import os
from pathlib import Path
import time
from typing import Any, Iterable, Mapping
from urllib.parse import quote
from urllib.request import Request, urlopen

BASE_URL = "https://openapi.foodsafetykorea.go.kr/api"
NO_DATA_CODES = {"INFO-200"}

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = cwd

print("프로젝트 경로:", PROJECT_ROOT)


프로젝트 경로: C:\Project_AI\mle-01-final-team3


## 1. 실행 설정

PowerShell에서 Jupyter를 실행하기 전에 아래처럼 인증키를 환경변수로 넣을 수 있습니다.

```powershell
$env:FOODSAFETYKOREA_API_KEY="발급받은_인증키"
```

- `sample`: 각 서비스의 앞부분만 시험 수집
- `incremental`: 최근 변경분 수집 (`CHNG_DT`, 최근 7일 범위)
- `full`: 전체 수집


In [2]:
from getpass import getpass

API_KEY = os.getenv("FOODSAFETYKOREA_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("식품안전나라 인증키를 입력하세요: ").strip()

RUN_COLLECTION = True
MODE = "full"  # 먼저 sample로 테스트
SELECTED_SERVICES = ["I1250", "C002"]

SAMPLE_SIZE = 100
PAGE_SIZE = 1000  # 식품안전나라 페이지당 최대 1,000건
CHANGED_SINCE = (date.today() - timedelta(days=6)).strftime("%Y%m%d")
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "foodsafetykorea"
PAUSE_SECONDS = 0.2
REQUEST_TIMEOUT = 60
MAX_RETRIES = 4

print({
    "mode": MODE,
    "services": SELECTED_SERVICES,
    "changed_since": CHANGED_SINCE,
    "output_dir": str(OUTPUT_DIR),
    "api_key_set": bool(API_KEY),
})


{'mode': 'full', 'services': ['I1250', 'C002'], 'changed_since': '20261116', 'output_dir': 'C:\\Project_AI\\mle-01-final-team3\\data\\raw\\foodsafetykorea', 'api_key_set': True}


## 2. 공통 수집 함수

API URL에는 인증키가 들어가므로 URL 자체를 로그로 출력하지 않습니다. 일시적인 연결 오류에는 지수 백오프로 재시도합니다.


In [3]:
def build_url(
    api_key: str,
    service_id: str,
    start: int,
    end: int,
    filters: Mapping[str, Any] | None = None,
    data_type: str = "json",
) -> str:
    if not api_key.strip():
        raise ValueError("API_KEY가 비어 있습니다.")
    if start < 1 or end < start:
        raise ValueError("start/end 범위를 확인하세요.")

    url = (
        f"{BASE_URL}/{quote(api_key.strip(), safe='')}/"
        f"{quote(service_id, safe='')}/{data_type}/{start}/{end}"
    )
    active_filters = {
        str(key): value
        for key, value in (filters or {}).items()
        if value is not None and str(value).strip()
    }
    if active_filters:
        encoded = "&".join(
            f"{quote(key, safe='')}={quote(str(value).strip(), safe='')}"
            for key, value in active_filters.items()
        )
        url = f"{url}/{encoded}"
    return url


def parse_payload(
    payload: Mapping[str, Any], service_id: str
) -> tuple[int, list[dict[str, Any]]]:
    service = payload.get(service_id)
    if not isinstance(service, Mapping):
        result = payload.get("RESULT") if isinstance(payload, Mapping) else None
        if isinstance(result, Mapping):
            code = str(result.get("CODE", "UNKNOWN"))
            message = str(result.get("MSG", "API 오류"))
            if code in NO_DATA_CODES:
                return 0, []
            raise RuntimeError(f"식품안전나라 API 오류 {code}: {message}")
        raise RuntimeError(f"응답에서 서비스 키 '{service_id}'를 찾지 못했습니다.")

    result = service.get("RESULT")
    if isinstance(result, Mapping):
        result_code = str(result.get("CODE", "UNKNOWN"))
        message = str(result.get("MSG", "API 오류"))
        if result_code in NO_DATA_CODES:
            return 0, []
        if result_code not in {"INFO-000", "0", "SUCCESS"}:
            raise RuntimeError(f"식품안전나라 API 오류 {result_code}: {message}")

    rows = service.get("row") or []
    if isinstance(rows, Mapping):
        rows = [dict(rows)]
    if not isinstance(rows, list):
        raise RuntimeError("API 응답의 row 형식이 예상과 다릅니다.")

    normalized = [dict(row) for row in rows if isinstance(row, Mapping)]
    total = int(service.get("total_count") or len(normalized))
    return total, normalized


def request_json(url: str, timeout: int = 60, retries: int = 4) -> Mapping[str, Any]:
    last_error: Exception | None = None
    for attempt in range(retries + 1):
        try:
            request = Request(url, headers={"User-Agent": "mle-food-safety-collector/1.0"})
            with urlopen(request, timeout=timeout) as response:
                raw = response.read().decode("utf-8-sig")
            payload = json.loads(raw)
            if not isinstance(payload, Mapping):
                raise RuntimeError("API JSON 최상위 형식이 객체가 아닙니다.")
            return payload
        except Exception as exc:
            last_error = exc
            if attempt >= retries:
                break
            wait_seconds = 2 ** attempt
            print(f"요청 실패 - {wait_seconds}초 후 재시도 ({attempt + 1}/{retries})")
            time.sleep(wait_seconds)
    raise RuntimeError("식품안전나라 API 요청에 반복 실패했습니다.") from last_error


def fetch_all(
    api_key: str,
    service_id: str,
    *,
    filters: Mapping[str, Any] | None = None,
    page_size: int = 1000,
    timeout: int = 60,
    retries: int = 4,
    pause_seconds: float = 0.2,
    max_records: int | None = None,
) -> list[dict[str, Any]]:
    if not 1 <= page_size <= 1000:
        raise ValueError("page_size는 1~1000 사이여야 합니다.")
    if max_records is not None and max_records < 1:
        raise ValueError("max_records는 1 이상이어야 합니다.")

    records: list[dict[str, Any]] = []
    start = 1
    while True:
        end = start + page_size - 1
        if max_records is not None:
            end = min(end, max_records)

        url = build_url(api_key, service_id, start, end, filters)
        payload = request_json(url, timeout=timeout, retries=retries)
        total, rows = parse_payload(payload, service_id)
        records.extend(rows)

        target = min(total, max_records) if max_records is not None else total
        print(f"{service_id}: {len(records):,}/{target:,}건 수집")
        if not rows or len(records) >= target:
            break

        start = end + 1
        if pause_seconds:
            time.sleep(pause_seconds)

    return records


def _fieldnames(records: Iterable[Mapping[str, Any]]) -> list[str]:
    fields: list[str] = []
    for record in records:
        for key in record:
            if key not in fields:
                fields.append(key)
    return fields


def save_records(
    records: list[dict[str, Any]],
    service_id: str,
    output_dir: str | Path,
    *,
    timestamp: str | None = None,
) -> dict[str, Path]:
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    suffix = timestamp or datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{service_id}_{suffix}"
    csv_path = destination / f"{stem}.csv"
    jsonl_path = destination / f"{stem}.jsonl"
    fields = _fieldnames(records)

    with csv_path.open("w", encoding="utf-8-sig", newline="") as stream:
        if fields:
            writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(records)

    with jsonl_path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")

    return {"csv": csv_path, "jsonl": jsonl_path}


## 3. 서비스 정의

세 서비스의 원본 필드명은 바꾸지 않고 저장합니다. I1250과 C002는 `PRDLST_REPORT_NO`로 1:N 결합할 수 있습니다.


In [4]:
SERVICES = {
    "I0030": "건강기능식품 품목제조 신고사항 현황",
    "I1250": "식품(첨가물) 품목제조보고",
    "C002": "식품(첨가물) 품목제조보고 원재료",
}

EXPECTED_KEY_FIELDS = {
    "I0030": ["PRDLST_REPORT_NO", "PRDLST_NM", "PRIMARY_FNCLTY", "RAWMTRL_NM"],
    "I1250": [
        "PRDLST_REPORT_NO", "PRDLST_NM", "PRDLST_DCNM",
        "HIENG_LNTRT_DVS_NM", "CHILD_CRTFC_YN",
    ],
    "C002": [
        "PRDLST_REPORT_NO", "PRDLST_NM", "PRDLST_DCNM",
        "RAWMTRL_NM", "RAWMTRL_ORDNO", "CHNG_DT",
    ],
}

unknown_services = sorted(set(SELECTED_SERVICES) - set(SERVICES))
if unknown_services:
    raise ValueError(f"지원하지 않는 서비스: {unknown_services}")
SERVICES


{'I0030': '건강기능식품 품목제조 신고사항 현황',
 'I1250': '식품(첨가물) 품목제조보고',
 'C002': '식품(첨가물) 품목제조보고 원재료'}

## 4. 수집 실행

설정을 확인한 뒤 `RUN_COLLECTION=True`로 바꾸고 실행하세요. 결과는 `data/raw/foodsafetykorea`에 UTF-8 BOM CSV와 JSONL로 저장됩니다.


In [5]:
if MODE not in {"sample", "incremental", "full"}:
    raise ValueError("MODE는 'sample', 'incremental', 'full' 중 하나여야 합니다.")
if RUN_COLLECTION and not API_KEY:
    raise ValueError("인증키가 없습니다. FOODSAFETYKOREA_API_KEY 환경변수를 설정하세요.")

collected: dict[str, list[dict[str, Any]]] = {}
saved_paths: dict[str, dict[str, Path]] = {}
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if RUN_COLLECTION:
    for service_id in SELECTED_SERVICES:
        filters = {"CHNG_DT": CHANGED_SINCE} if MODE == "incremental" else None
        max_records = SAMPLE_SIZE if MODE == "sample" else None
        print(f"\n[{service_id}] {SERVICES[service_id]}")
        rows = fetch_all(
            API_KEY,
            service_id,
            filters=filters,
            page_size=PAGE_SIZE,
            timeout=REQUEST_TIMEOUT,
            retries=MAX_RETRIES,
            pause_seconds=PAUSE_SECONDS,
            max_records=max_records,
        )
        collected[service_id] = rows
        saved_paths[service_id] = save_records(
            rows,
            service_id,
            OUTPUT_DIR,
            timestamp=run_timestamp,
        )
        if rows:
            missing = [
                field for field in EXPECTED_KEY_FIELDS[service_id]
                if field not in rows[0]
            ]
            if missing:
                print("주의 - 예상 필드 중 응답에 없는 필드:", missing)
else:
    print("RUN_COLLECTION=False: 네트워크 수집을 건너뛰었습니다.")

saved_paths



[I1250] 식품(첨가물) 품목제조보고
I1250: 1,000/1,087,964건 수집
I1250: 2,000/1,087,964건 수집
I1250: 3,000/1,087,964건 수집
I1250: 4,000/1,087,964건 수집
I1250: 5,000/1,087,964건 수집
I1250: 6,000/1,087,964건 수집
I1250: 7,000/1,087,964건 수집
I1250: 8,000/1,087,964건 수집
I1250: 9,000/1,087,964건 수집
I1250: 10,000/1,087,964건 수집
I1250: 11,000/1,087,964건 수집
I1250: 12,000/1,087,964건 수집
I1250: 13,000/1,087,964건 수집
I1250: 14,000/1,087,964건 수집
I1250: 15,000/1,087,964건 수집
I1250: 16,000/1,087,964건 수집
I1250: 17,000/1,087,964건 수집
I1250: 18,000/1,087,964건 수집
I1250: 19,000/1,087,964건 수집
I1250: 20,000/1,087,964건 수집
I1250: 21,000/1,087,964건 수집
I1250: 22,000/1,087,964건 수집
I1250: 23,000/1,087,964건 수집
I1250: 24,000/1,087,964건 수집
I1250: 25,000/1,087,964건 수집
I1250: 26,000/1,087,964건 수집
I1250: 27,000/1,087,964건 수집
I1250: 28,000/1,087,964건 수집
I1250: 29,000/1,087,964건 수집
I1250: 30,000/1,087,964건 수집
I1250: 31,000/1,087,964건 수집
I1250: 32,000/1,087,964건 수집
I1250: 33,000/1,087,964건 수집
I1250: 34,000/1,087,964건 수집
I1250: 35,000/1,087,964건 수집
I1250

RuntimeError: 식품안전나라 API 오류 INFO-320: CHNG_DT 파라미터를 사용하지 않으면 1시간당 100회까지만 호출할 수 있습니다.

## 5. 결과 점검

- I0030의 기능성 문구는 제품 신고 데이터이므로 법적 허용 문구의 단독 근거로 쓰지 않습니다.
- I1250과 C002는 `PRDLST_REPORT_NO`로 결합합니다.
- C002의 원재료명과 표시순서는 원본 그대로 보존한 뒤 별도 정규화 테이블에서 표준화합니다.


In [ ]:
for service_id, rows in collected.items():
    print(f"\n{service_id}: {len(rows):,}건")
    if not rows:
        continue
    print("실제 필드:", list(rows[0]))
    preview_fields = [
        field for field in EXPECTED_KEY_FIELDS[service_id]
        if field in rows[0]
    ]
    print("첫 행 주요 값:", {field: rows[0].get(field) for field in preview_fields})


## 운영 권장사항

1. 처음에는 `sample`로 세 서비스의 응답 필드를 확인합니다.
2. 최초 기준 스냅샷은 `full`로 한 번 수집합니다.
3. 이후에는 매주 `incremental`로 최근 7일 변경분을 저장합니다.
4. 원본 파일은 수정하지 말고 적재 시각이 포함된 파일명으로 보존합니다.
5. 제품 데이터와 시행일이 관리된 법령·고시 테이블을 분리하고, 판정 시 함께 조회합니다.
